# Illucidate Quick Start

**AI-powered early detection for bacterial strain classification**

This notebook demonstrates:
1. Installing Illucidate
2. Loading growth curve data (auto-detects format)
3. Running early detection analysis
4. Visualizing results

## 1. Installation

**For Colab:**

In [ ]:
# If running in Colab, clone the repository
import sys
import os

if 'google.colab' in sys.modules:
    # Clone repo
    if not os.path.exists('illucidate'):
        !git clone https://github.com/yourusername/illucidate.git  # Update with your repo
    
    # Install in development mode
    %cd illucidate
    !pip install -e .
else:
    print("Running locally - make sure illucidate is installed or in PYTHONPATH")

## 2. Import Illucidate

In [ ]:
from illucidate import load_data, EarlyDetectionAnalyzer
import pandas as pd
import matplotlib.pyplot as plt

print("✓ Illucidate imported successfully!")

## 3. Load Sample Data

Illucidate automatically detects the format of your data file!

In [ ]:
# Example: Load E. coli dataset from Figshare
# Download from: https://springernature.figshare.com/articles/dataset/5918608

# Option 1: Upload your own file
from google.colab import files
uploaded = files.upload()
data_file = list(uploaded.keys())[0]

# Option 2: Or specify a file path
# data_file = 'KHKgrowthcurves_LB.xlsx'

In [ ]:
# Load data (auto-detects format)
dataset = load_data(data_file, measurement_type='OD600')

# Show summary
print(dataset.summary())

## 4. Convert to Wide Format

The analyzer expects data in wide format (time × samples)

In [ ]:
# Convert to wide format
wide_df = dataset.to_wide_format()

print(f"Data shape: {wide_df.shape[0]} timepoints × {wide_df.shape[1]} samples")
wide_df.head()

## 5. Visualize Growth Curves

In [ ]:
# Plot first 6 samples
fig, ax = plt.subplots(figsize=(12, 6))

sample_ids = dataset.get_sample_ids()
for i, sample_id in enumerate(sample_ids[:6]):
    sample_data = dataset.get_sample(sample_id)
    
    # Plot each replicate
    for rep in sample_data['replicate'].unique():
        rep_data = sample_data[sample_data['replicate'] == rep]
        alpha = 0.3 if rep > 1 else 1.0
        linewidth = 2 if rep == 1 else 1
        ax.plot(rep_data['time'], rep_data['value'], 
               alpha=alpha, linewidth=linewidth,
               label=f"{sample_id}" if rep == 1 else "")

ax.set_xlabel('Time (h)', fontsize=12)
ax.set_ylabel('OD600', fontsize=12)
ax.set_title('Growth Curves (First 6 Samples)', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Create Sample Labels

For classification, create labels for your samples

In [ ]:
# Example: Create 4 groups
sample_ids = dataset.get_sample_ids()
n_samples = len(sample_ids)
samples_per_group = n_samples // 4

sample_labels = {}
for i, sample_id in enumerate(sample_ids):
    if i < samples_per_group:
        sample_labels[sample_id] = 'Group_A'
    elif i < 2 * samples_per_group:
        sample_labels[sample_id] = 'Group_B'
    elif i < 3 * samples_per_group:
        sample_labels[sample_id] = 'Group_C'
    else:
        sample_labels[sample_id] = 'Group_D'

print(f"Created {len(set(sample_labels.values()))} groups:")
for group in sorted(set(sample_labels.values())):
    count = sum(1 for v in sample_labels.values() if v == group)
    print(f"  {group}: {count} samples")

## 7. Run Early Detection Analysis

The analyzer extracts hundreds of features and finds early indicators

In [ ]:
# Initialize analyzer
analyzer = EarlyDetectionAnalyzer(wide_df, sample_labels)

# Run analysis
results = analyzer.analyze()

## 8. View Results

In [ ]:
# Generated features
print(f"Total features generated: {len(results['features'].columns)}")
print("\nFirst few features:")
results['features'].head()

In [ ]:
# Early divergence signals
print("Top 5 Early Divergence Signals:")
print("="*70)

for i, signal in enumerate(results['early_divergence'][:5], 1):
    print(f"\n{i}. {signal['feature']}")
    print(f"   P-value: {signal['p_value']:.4e}")
    print(f"   Effect size: {signal['effect_size']:.3f}")
    print(f"   Group 1 mean: {signal['group1_mean']:.3f}")
    print(f"   Group 2 mean: {signal['group2_mean']:.3f}")

## 9. Generate Report

In [ ]:
# Generate comprehensive report
report = analyzer.generate_report()
print(report)

## 10. Save Results

In [ ]:
# Save features to CSV
results['features'].to_csv('features.csv', index=False)

# Save report to text file
with open('analysis_report.txt', 'w') as f:
    f.write(report)

print("✓ Results saved!")

# Download files (Colab only)
if 'google.colab' in sys.modules:
    files.download('features.csv')
    files.download('analysis_report.txt')

## Next Steps

1. **Try with your own data**: Upload your Excel or CSV files
2. **Adjust labels**: Create meaningful sample groups
3. **Explore features**: Check which features show early divergence
4. **Cross-media validation**: Test with different growth media

---

**Questions or issues?** Check the documentation or open an issue on GitHub!